# 🔍 Grouped-Query Attention (GQA): Complete Hands-On Lab

Welcome! In this notebook, we explore <b>Grouped-Query Attention (GQA)</b>—the attention architecture used in <b>Llama 3, Mistral, and Mirendil-Base</b> that cuts Key-Value memory by up to <b>87.5%</b> while preserving model accuracy.

---

## 1. The Core Metaphor: The Office Library

Imagine an AI research office with <b>12 Analysts</b> (Query heads), each looking for different code patterns.<br>
To do their jobs, they must look up past tokens stored in reference cabinets (<b>Key / Value Cache</b>).

<pre style='font-size: 14px; background: #0d1117; color: #c9d1d9; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
1. MHA (Old Way - Multi-Head Attention):
   Every single Analyst keeps their own private copy of the cabinet!
   12 Analysts = 12 Filing Cabinets (100% Memory Cost - GPU Runs Out of RAM!)

2. MQA (The Extreme Cheap Way - Multi-Query Attention):
   All 12 Analysts share only 1 single filing cabinet.
   (Saves 92% memory, but model intelligence drops noticeably!)

3. GQA (The Modern Winner - Grouped-Query Attention):
   Group the 12 analysts into 4 pods of 3. Each pod shares 1 cabinet!
   12 Analysts &rarr; 4 Cabinets (66.7% Memory Saved with virtually 0 quality loss!)
</pre>

---
## 2. Visual Architecture Diagram

<pre style='font-size: 14px; background: #0d1117; color: #58a6ff; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
Query Heads (12 Total)               Key / Value Heads (4 Total)
┌─────────────────────────┐         ┌─────────────────────────┐
│ Pod 1:  [ Q0,  Q1,  Q2 ]│ ──────> │        [ KV0 ]          │  (Shared)
├─────────────────────────┤         ├─────────────────────────┤
│ Pod 2:  [ Q3,  Q4,  Q5 ]│ ──────> │        [ KV1 ]          │  (Shared)
├─────────────────────────┤         ├─────────────────────────┤
│ Pod 3:  [ Q6,  Q7,  Q8 ]│ ──────> │        [ KV2 ]          │  (Shared)
├─────────────────────────┤         ├─────────────────────────┤
│ Pod 4:  [ Q9, Q10, Q11 ]│ ──────> │        [ KV3 ]          │  (Shared)
└─────────────────────────┘         └─────────────────────────┘
</pre>

---
## 3. Exercise 1: The Repeat-Interleave Mechanism on Paper 📝

When computing attention scores in PyTorch, Query has <b>12 heads</b>, but Key has only <b>4 heads</b>.<br>
To compute dot-product attention in parallel, PyTorch duplicates each of the 4 KV heads <b>3 times</b>.<br>
This operation is called <code>repeat_interleave(3, dim=1)</code>.

### Your Turn on Paper:
Suppose our 4 Key heads are labeled:
<pre style='font-size: 15px; background: #161b22; color: #bc8cff; padding: 12px; border-radius: 8px;'>
K = [ KV0,  KV1,  KV2,  KV3 ]
</pre>

If we repeat each element <b>3 times consecutively</b>, what is the resulting 12-head list?
<br><br>
<b>Step-by-Step on Paper:</b>
1. Repeat KV0 3 times: [ KV0, KV0, KV0 ]
2. Repeat KV1 3 times: [ KV1, KV1, KV1 ]
3. Repeat KV2 3 times: [ KV2, KV2, KV2 ]
4. Repeat KV3 3 times: [ KV3, KV3, KV3 ]

<br>
<b>Expanded 12-Head List:</b><br>
<span style='color: #3fb950; font-size: 15px; font-weight: bold;'>
[ KV0, KV0, KV0,   KV1, KV1, KV1,   KV2, KV2, KV2,   KV3, KV3, KV3 ]
</span><br><br>
Now the 12 Query heads have a matching Key partner for 100% parallel matrix multiplication on GPU Tensor Cores!

In [ ]:
import torch

# 4 KV heads, each with 1 simple feature for demonstration
kv_heads = torch.tensor([[10], [20], [30], [40]])  # Shape: [4 heads, 1]
print(f"Original KV heads (4 heads):\n{kv_heads.squeeze().tolist()}")

# Repeat each head 3 times to match 12 Query heads
expanded_kv = kv_heads.repeat_interleave(3, dim=0)
print(f"\nExpanded KV heads (12 heads):\n{expanded_kv.squeeze().tolist()}")

assert len(expanded_kv) == 12
print("\nSUCCESS: Interleave repetition verified!")

---
## 4. Exercise 2: Memory Savings Calculation on Paper 📝

Let's calculate the real-world GPU RAM savings for a sequence of <b>4,096 tokens</b>.
<br><br>
Suppose:
- Sequence length = 4,096 tokens
- Head dimension = 128 numbers per head
- Precision = 2 bytes per number (bfloat16)

<pre style='font-size: 15px; background: #161b22; color: #d29922; padding: 12px; border-radius: 8px;'>
Bytes per head = 4096 tokens · 128 dims · 2 bytes = 1,048,576 bytes ≈ 1 MB per head
(Both Key and Value = 2 MB per head pair)
</pre>

### On Paper:
1. <b>Under MHA (32 KV heads):</b><br>
   32 heads · 2 MB = <b>64 MB per layer</b>.<br>
   Across 32 layers = <b>2,048 MB (2.0 GB)</b> per single user prompt!
<br><br>
2. <b>Under GQA (8 KV heads):</b><br>
   8 heads · 2 MB = <b>16 MB per layer</b>.<br>
   Across 32 layers = <b>512 MB (0.5 GB)</b> per user prompt!
<br><br>
<b>Memory Reduction: 75% of KV Cache RAM saved!</b>

---
## 5. Verify in PyTorch with Real Project GQA Code

Now let's trace a forward pass through our project's actual `GroupedQueryAttention` class from `src/model/attention.py`:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
from src.model.attention import GroupedQueryAttention
from src.model.rope import precompute_freqs_cis

# Setup GQA with 12 Query heads and 4 KV heads
dim = 768
n_heads = 12
n_kv_heads = 4
seq_len = 8

gqa = GroupedQueryAttention(dim=dim, n_heads=n_heads, n_kv_heads=n_kv_heads)
cos, sin = precompute_freqs_cis(dim // n_heads, max_seq_len=seq_len)

print(f"Q projection matrix shape:  {list(gqa.q_proj.weight.shape)} (12 heads)")
print(f"K projection matrix shape:  {list(gqa.k_proj.weight.shape)}  (4 heads - 66.7% smaller!)")
print(f"V projection matrix shape:  {list(gqa.v_proj.weight.shape)}  (4 heads - 66.7% smaller!)")

# Forward pass on dummy batch
x = torch.randn(2, seq_len, dim)
out = gqa(x, cos, sin)

print(f"\nInput Tensor Shape:        {list(x.shape)}")
print(f"Attention Output Shape:    {list(out.shape)}")
assert out.shape == x.shape
print("\nSUCCESS: GQA Forward pass verified completely!")